# Agent evaluation (Phases 9–10): repair agent vs agent baselines (GPU)

Runs agent systems over the R / C / T scenarios (`benchmark/agent_scenarios/rct_v4.jsonl`, built from pool v4: all four datasets) and the E / S scenarios (`benchmark/agent_scenarios/{E,S}/`) with one
real model (same model, limits and budget for every system), and judges each final output against the reference.
Scoring runs on the laptop: download `agent_bundle_<stage>.zip` (one file, link printed by the last cell), unzip it at the project root, then `python eval/agent_metrics.py results/stage2_agent dev`. Each system folder also has `traces.jsonl` (every model reply), for checking the agent's actions.

**Step 1 (`STAGE = "dev"`):** full agent on the dev scenarios only (14 R + 20 C + 6 T + 3 E + 3 S), to check the agent prompt with the real model
(README Phase 9, task 5). **Step 2 (`STAGE = "test"`):** all systems on the test scenarios.
Everything is resumable and cached; everything goes into one `agent_bundle_<stage>.zip`.

Settings: GPU T4 x1, Internet on, secret `GITHUB_TOKEN` for a private repo. *Save Version → Save & Run All*.

**GPU time (estimate):** about 15 s per agent step for the 7B in 4-bit, so ~3 h per system on test (83 R + 24 T + 15 E/S; C is nearly free with the oracle verifier). Run one system per session if needed: it resumes.

In [ ]:
# ---------------- Configuration ----------------
REPO_URL = "https://github.com/<user>/<repo>.git"   # <- your repo
BRANCH = "main"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"; LOAD_IN_4BIT = True     # one model for every agent system
STAGE = "dev"                 # "dev" first (prompt check), then "test"
VERIFIER = "oracle"           # oracle | parsed:<path to parses.jsonl> | conventional
# (system name, mode, extra flags). Main systems first; ablations are optional (R scenarios only, to save GPU time).
SYSTEMS = [
    ("agent_full",     "full",     []),
    ("agent_feedback", "feedback", []),
    ("agent_naive",    "naive",    []),
]
ABLATIONS = [
    ("agent_full_no_memory",    "full", ["--no-memory"]),
    ("agent_full_coarse",       "full", ["--coarse-report"]),
    ("agent_full_no_contract",  "full", ["--hide-contract"]),
    ("agent_full_no_escalate",  "full", ["--no-escalate"]),
]
RUN_ABLATIONS = False
if STAGE == "dev":
    SYSTEMS, RUN_ABLATIONS = SYSTEMS[:1], False
OUT = "/kaggle/working/results/stage2_agent"   # agent prompt a1-v4, scenarios rct_v4
CACHE = "/kaggle/working/llm_cache"
print(STAGE, [s[0] for s in SYSTEMS], "ablations" if RUN_ABLATIONS else "")

In [ ]:
# ---------------- Get the code ----------------
import os, shutil, subprocess, time, json
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    print("no GITHUB_TOKEN secret; cloning without a token (public repo)")
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/kaggle/working/repo"], check=True)
os.chdir("/kaggle/working/repo")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
NEEDED = ["eval/runner_agent.py", "src/igt/agents/repair_agent.py", "benchmark/agent_scenarios/rct_v4.jsonl",
          "benchmark/pool/pool_v4.jsonl", "src/igt/llm/hf.py"]
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f"missing (push them first): {missing}"
print("commit", COMMIT)

In [ ]:
# ---------------- Libraries: pinned pandas for the sandbox + bitsandbytes ----------------
# The sandbox and verifier need pandas 2.3.3 / numpy 2.1.3 (pickles, dtype rules). Install them into a separate folder
# and use it only for this repo's processes, so Kaggle's torch stack stays untouched.
subprocess.run(["pip", "install", "-q", "--target", "/kaggle/working/evalpkgs", "pandas==2.3.3", "numpy==2.1.3",
                "pyyaml==6.0.2", "pydantic==2.14.1"], check=True)
if LOAD_IN_4BIT:
    subprocess.run(["pip", "install", "-q", "bitsandbytes"], check=True)
ENV = dict(os.environ, PYTHONPATH="/kaggle/working/evalpkgs")
# pip may print "ERROR: pip's dependency resolver ..." about Kaggle's pre-installed packages (bigframes, gradio, ...).
# That is only a warning: they aren't used, and the pinned versions live in their own folder.
probe = ("import pandas, numpy, torch, transformers; "
         "print('pandas', pandas.__version__, '| numpy', numpy.__version__, '| torch', torch.__version__, "
         "'| transformers', transformers.__version__, '| GPU', torch.cuda.is_available())")
if LOAD_IN_4BIT:
    probe += "; import bitsandbytes; print('bitsandbytes', bitsandbytes.__version__)"
p = subprocess.run(["python", "-c", probe], env=ENV, capture_output=True, text=True)
print(p.stdout, p.stderr[-1500:])
assert p.returncode == 0 and "pandas 2.3.3" in p.stdout and "numpy 2.1.3" in p.stdout and "GPU True" in p.stdout, \
    "environment check failed: see the output above"

In [ ]:
# ---------------- Run ----------------
LOGS = f"{OUT}/logs"
os.makedirs(LOGS, exist_ok=True)

def run(system, mode, flags, categories):
    cmd = ["python", "eval/runner_agent.py", "--system", system, "--mode", mode, "--backend", "hf",
           "--model", MODEL_ID, "--verifier", VERIFIER, "--splits", STAGE, "--categories", *categories,
           "--cache-dir", CACHE, "--out", OUT, *flags] + (["--load-in-4bit"] if LOAD_IN_4BIT else [])
    t0 = time.time()
    p = subprocess.run(cmd, env=ENV, capture_output=True, text=True)
    with open(f"{LOGS}/{system}_{STAGE}.log", "a") as fh:          # full stdout/stderr, kept in the bundle
        fh.write(f"$ {' '.join(cmd)}\n--- stdout ---\n{p.stdout}\n--- stderr ---\n{p.stderr}\n"
                 f"--- exit {p.returncode}, {(time.time() - t0) / 60:.1f} min ---\n")
    print(p.stdout[-2500:]); print(p.stderr[-2500:])
    assert p.returncode == 0, f"{system} failed: see {LOGS}/{system}_{STAGE}.log"
    print(f"== {system} done in {(time.time() - t0) / 60:.1f} min")

for system, mode, flags in SYSTEMS:
    run(system, mode, flags, ["R", "C", "T", "E", "S"])
if RUN_ABLATIONS:
    for system, mode, flags in ABLATIONS:
        run(system, mode, flags, ["R"])

In [ ]:
# ---------------- Summary + ONE zip with everything needed on the laptop ----------------
# The zip mirrors the project layout: unzip it at the project root and the results land in
# results/stage2_agent/<system>/ (runs.jsonl, traces.jsonl, run_info.json, git_commit.txt), plus
# results/stage2_agent/agent_metrics_<stage>.json, metrics_<stage>.txt, logs/ and bundle_info.json.
import zipfile, datetime
INCLUDE_CACHE = False     # True: also add the LLM cache (large; only needed to resume this run elsewhere)
CLEANUP_REPO = False      # True for "Save Version" runs; keep False in an interactive session so cells can be rerun

p = subprocess.run(["python", "eval/agent_metrics.py", OUT, STAGE], env=ENV, capture_output=True, text=True)
print(p.stdout[-4000:], p.stderr[-1500:])
open(f"{OUT}/metrics_{STAGE}.txt", "w").write(p.stdout + "\n" + p.stderr)

systems = sorted(d for d in os.listdir(OUT) if os.path.isdir(f"{OUT}/{d}") and d != "logs")
for d in systems:
    open(f"{OUT}/{d}/git_commit.txt", "w").write(COMMIT)
info = dict(stage=STAGE, commit=COMMIT, model=MODEL_ID, load_in_4bit=LOAD_IN_4BIT, verifier=VERIFIER,
            systems=systems, created=datetime.datetime.now().isoformat(timespec="seconds"),
            runs={d: sum(1 for _ in open(f"{OUT}/{d}/runs.jsonl")) for d in systems if os.path.exists(f"{OUT}/{d}/runs.jsonl")})
json.dump(info, open(f"{OUT}/bundle_info.json", "w"), indent=2)

BUNDLE = f"/kaggle/working/agent_bundle_{STAGE}.zip"
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for f in files:
            full = os.path.join(root, f)
            z.write(full, os.path.join("results/stage2_agent", os.path.relpath(full, OUT)))
    if INCLUDE_CACHE and os.path.isdir(CACHE):
        for root, _, files in os.walk(CACHE):
            for f in files:
                full = os.path.join(root, f)
                z.write(full, os.path.join("llm_cache", os.path.relpath(full, CACHE)))
    names = z.namelist()
print(f"saved {BUNDLE}: {len(names)} files, {os.path.getsize(BUNDLE) / 1e6:.1f} MB")
print("\n".join("  " + n for n in names if not n.startswith("llm_cache/")))
print(json.dumps(info, indent=2))
try:
    from IPython.display import FileLink, display
    os.chdir("/kaggle/working")
    display(FileLink(os.path.basename(BUNDLE)))       # click to download in an interactive session
except Exception:
    pass
if CLEANUP_REPO:
    os.chdir("/kaggle/working")
    shutil.rmtree("/kaggle/working/repo", ignore_errors=True)